In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import pickle

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [2]:
# Sample data
data = {
    'text': [
        'This is a SAMPLE text for processing! It has punctuation.',
        'Another Example with NUMBERS 123 and special chars @#$',
        'Machine learning is amazing and powerful',
        'Natural Language Processing helps computers understand text'
    ],
    'label': ['positive', 'negative', 'positive', 'neutral']
}

df = pd.DataFrame(data)
print("Original Data:")
print(df)

Original Data:
                                                text     label
0  This is a SAMPLE text for processing! It has p...  positive
1  Another Example with NUMBERS 123 and special c...  negative
2           Machine learning is amazing and powerful  positive
3  Natural Language Processing helps computers un...   neutral


In [3]:
# Text Cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['text'].apply(clean_text)
print("\nAfter Text Cleaning:")
print(df[['text', 'cleaned_text']])


After Text Cleaning:
                                                text  \
0  This is a SAMPLE text for processing! It has p...   
1  Another Example with NUMBERS 123 and special c...   
2           Machine learning is amazing and powerful   
3  Natural Language Processing helps computers un...   

                                        cleaned_text  
0  this is a sample text for processing it has pu...  
1     another example with numbers and special chars  
2           machine learning is amazing and powerful  
3  natural language processing helps computers un...  


In [4]:
# Remove Stop Words
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)

df['no_stopwords'] = df['cleaned_text'].apply(remove_stopwords)
print("\nAfter Removing Stop Words:")
print(df[['cleaned_text', 'no_stopwords']])


After Removing Stop Words:
                                        cleaned_text  \
0  this is a sample text for processing it has pu...   
1     another example with numbers and special chars   
2           machine learning is amazing and powerful   
3  natural language processing helps computers un...   

                                        no_stopwords  
0                 sample text processing punctuation  
1              another example numbers special chars  
2                  machine learning amazing powerful  
3  natural language processing helps computers un...  


In [5]:
# Lemmatization
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = text.split()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(lemmatized_words)

df['lemmatized_text'] = df['no_stopwords'].apply(lemmatize_text)
print("\nAfter Lemmatization:")
print(df[['no_stopwords', 'lemmatized_text']])


After Lemmatization:
                                        no_stopwords  \
0                 sample text processing punctuation   
1              another example numbers special chars   
2                  machine learning amazing powerful   
3  natural language processing helps computers un...   

                                     lemmatized_text  
0                 sample text processing punctuation  
1                another example number special char  
2                  machine learning amazing powerful  
3  natural language processing help computer unde...  


In [6]:
# Label Encoding
label_encoder = LabelEncoder()
df['encoded_label'] = label_encoder.fit_transform(df['label'])
print("\nAfter Label Encoding:")
print(df[['label', 'encoded_label']])


After Label Encoding:
      label  encoded_label
0  positive              2
1  negative              0
2  positive              2
3   neutral              1


In [7]:
# TF-IDF Representation
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['lemmatized_text'])

print("\nTF-IDF Matrix Shape:", tfidf_matrix.shape)
print("\nTF-IDF Feature Names:", tfidf_vectorizer.get_feature_names_out())
print("\nTF-IDF Matrix:")
print(tfidf_matrix.toarray())


TF-IDF Matrix Shape: (4, 18)

TF-IDF Feature Names: ['amazing' 'another' 'char' 'computer' 'example' 'help' 'language'
 'learning' 'machine' 'natural' 'number' 'powerful' 'processing'
 'punctuation' 'sample' 'special' 'text' 'understand']

TF-IDF Matrix:
[[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.43779123 0.55528266 0.55528266 0.         0.43779123 0.        ]
 [0.         0.4472136  0.4472136  0.         0.4472136  0.
  0.         0.         0.         0.         0.4472136  0.
  0.         0.         0.         0.4472136  0.         0.        ]
 [0.5        0.         0.         0.         0.         0.
  0.         0.5        0.5        0.         0.         0.5
  0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.40021825 0.         0.40021825
  0.40021825 0.         0.         0.40021825 0.         0.
  0.31553666 0.         0.         0.         0.

In [11]:
# Save Outputs
import os
print("Current working directory:", os.getcwd())

df.to_csv('processed_data.csv', index=False)
print("Processed data saved to 'processed_data.csv'")

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print("TF-IDF vectorizer saved to 'tfidf_vectorizer.pkl'")

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("Label encoder saved to 'label_encoder.pkl'")

with open('tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)
print("TF-IDF matrix saved to 'tfidf_matrix.pkl'")

print("\nFiles saved in:", os.getcwd())
print("\nFiles in directory:")
for file in os.listdir('.'):
    if file.endswith(('.csv', '.pkl')):
        print(f"  - {file}")

Current working directory: /content
Processed data saved to 'processed_data.csv'
TF-IDF vectorizer saved to 'tfidf_vectorizer.pkl'
Label encoder saved to 'label_encoder.pkl'
TF-IDF matrix saved to 'tfidf_matrix.pkl'

Files saved in: /content

Files in directory:
  - processed_data.csv
  - E:\NLP_assignments\tfidf_matrix.pkl
  - tfidf_matrix.pkl
  - E:\NLP_assignments\label_encoder.pkl
  - E:\NLP_assignments\tfidf_vectorizer.pkl
  - E:\NLP_assignments\processed_data.csv
  - label_encoder.pkl
  - tfidf_vectorizer.pkl


In [13]:
# Download files (if running in Colab or similar environment)
try:
    from google.colab import files
    files.download('processed_data.csv')
    files.download('tfidf_vectorizer.pkl')
    files.download('label_encoder.pkl')
    files.download('tfidf_matrix.pkl')
    print("Files downloaded to your local machine")
except:
    print("Files are saved in the current directory:", os.getcwd())
    print("If running locally, copy them from there to your desired location")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Files downloaded to your local machine
